# TP Panda — MuJoCo dans Google Colab

**Groupe 1 : un Panda, une table, le pot French’s (YCB mustard bottle).**

Ce notebook autonome installe MuJoCo, télécharge les modèles, crée la scène, résout une
cinématique inverse de pose **position + orientation**, puis ferme la pince avec une
**loi d’impédance en force**, soulève le pot et le maintient trois secondes en hauteur.
La simulation s’affiche sous forme de vidéo intégrée.

Dans Colab : **Fichier → Importer un notebook**, sélectionner ce fichier, puis
**Exécution → Tout exécuter**. Un environnement **CPU** suffit : le rendu OSMesa est logiciel.
Exécuter dans une session neuve pour configurer OpenGL avant le premier import de MuJoCo.
Les téléchargements demandent une connexion Internet. Aucun compte ou jeton n’est nécessaire.

La trajectoire de démonstration utilise une pose connue du pot. Contact-GraspNet n’est pas exécuté ici.
La cinématique inverse n’est pas un planificateur avec évitement d’obstacles : si vous changez
la cible, vérifiez la trajectoire et les collisions. Le TP mesure le déplacement du pot,
sa séparation de la table et son glissement pendant le maintien en hauteur.


In [ ]:
# Installation dans la machine Colab (pas sur votre ordinateur).
import subprocess, sys, os
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'libosmesa6'], check=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'mujoco==3.3.7', 'numpy>=1.26,<3', 'scipy>=1.11,<2', 'requests>=2.31,<3',
    'imageio>=2.34,<3', 'imageio-ffmpeg==0.6.0', 'pillow>=10', 'matplotlib>=3.7'])
os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'
print('Installation terminée. Backend : OSMesa (CPU).')

## 1. Modèles publics et téléchargements

- [Panda MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie/tree/main/franka_emika_panda), version figée dans le code.
- [Pot French’s YCB, modèle mustard_bottle](https://github.com/elpis-lab/YCB_Dataset/blob/main/ycb/mustard_bottle.xml), maillage texturé et trois maillages de collision convexes. Dimensions conservées (environ 5,8 × 9,6 × 19,1 cm), masse du fichier source : 0,603 kg.
- [Votre Panda Hydrax](https://github.com/MaximeSabbah/hydrax/blob/feedback-mppi-panda/hydrax/models/panda/panda.xml) : son site `gripper` est attaché à `hand` à `pos="0 0 0.1"`. Nous reprenons cet offset pour `grasp_center`.

La scène utilise **Menagerie**, avec ses collisions actives, et remplace les actionneurs
de position par des moteurs de couple/force. Les licences des deux modèles sont téléchargées
avec les fichiers. Les inerties et les contacts restent des approximations de simulation.


In [ ]:
%%writefile fetch_assets.py
"""Download only the Panda and French's YCB assets used by this exercise."""
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import xml.etree.ElementTree as ET
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

PANDA_REV = "c96a32d28fb5da84da38c1da4d749e7a13212855"
YCB_REV = "9e8c6488a2ff673d9aa48a91492fb89423c1b106"

def download(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=1)))
    response = session.get(url, timeout=90)
    response.raise_for_status()
    destination.write_bytes(response.content)
    return destination

def fetch_assets(root="assets"):
    root = Path(root).resolve()
    base = f"https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/{PANDA_REV}/franka_emika_panda/"
    xml = download(base + "panda.xml", root / "panda/panda.xml")
    meshes = {node.get("file") for node in ET.parse(xml).findall(".//mesh") if node.get("file")}
    jobs = [(base + "assets/" + name, root / "panda/assets" / name) for name in meshes]
    jobs += [(base + "LICENSE", root / "panda/LICENSE")]
    ycb = f"https://raw.githubusercontent.com/elpis-lab/YCB_Dataset/{YCB_REV}/"
    jobs += [(ycb + "ycb/mustard_bottle.xml", root / "mustard/source.xml")]
    jobs += [(ycb + "LICENSE", root / "mustard/LICENSE")]
    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(lambda pair: download(*pair), jobs))
    # Follow the source XML so paths do not depend on guesses about the dataset layout.
    source = ET.parse(root / "mustard/source.xml")
    jobs = []
    for node in source.findall(".//mesh") + source.findall(".//texture"):
        name = node.get("file")
        if name:
            jobs.append((ycb + "ycb/" + name, root / "mustard" / name))
    with ThreadPoolExecutor(max_workers=4) as pool:
        list(pool.map(lambda pair: download(*pair), jobs))
    print("Assets ready:", root)
    return root

if __name__ == "__main__":
    fetch_assets()


In [ ]:
from fetch_assets import fetch_assets
asset_root = fetch_assets()

## 2. Scène et contrôleur Python

Le code ci-dessous est enregistré dans `panda_lab.py` : vous pouvez le modifier directement.
La table a son plateau à z = 0,4 m. Le robot est fixé sur le plateau ; le pot est libre
et soumis à la gravité et aux contacts.

**Repères et unités** : mètres, radians, secondes, newtons, newtons-mètres.
`hand` est l’origine de la main ; `grasp_center` est notre TCP, au centre nominal entre les doigts.
Son orientation est celle de `hand`. +z pointe de la main vers les doigts ; l’axe y est
l’axe d’ouverture. Les axes affichés sont x rouge, y vert, z bleu.

Le TCP est fixe dans la main : ce n’est pas un point de contact mesuré et il ne suit pas
une éventuelle asymétrie des doigts. La matrice `T_world_tcp` transforme des coordonnées
du TCP vers le monde. Son inverse transforme du monde vers le TCP.

Pour les doigts, avec une consigne par doigt q_d en mètres :

$$F_i = \mathrm{clip}\left(K(q_d-q_i)-D\dot q_i,\;-F_{max},\;F_{max}\right).$$

Valeurs initiales : K = 400 N/m, D = 8 N·s/m, Fmax = 12 N **par doigt**.
q_d = 0,04 ouvre ; q_d = 0 ferme. L’ouverture nominale totale est q₁ + q₂.
La consigne diminue progressivement et le contact arrête les doigts avant q = 0.
Il s’agit d’une impédance articulaire des doigts ; la saturation de commande ne garantit
pas une borne identique sur les forces transitoires de contact, qui sont mesurées séparément.

Le bras reçoit un PD articulaire avec compensation des efforts de biais MuJoCo.
L’IK utilise le Jacobien du TCP et une pseudo-inverse amortie ; elle ne modifie pas l’état
du robot simulé. Les mouvements suivants sont intégrés par la dynamique MuJoCo.


**Contacts pour le soulèvement.** Le solveur utilise `cone='elliptic'`, `impratio=10`
et `noslip_iterations=20` pour réduire la dérive tangentielle des contacts souples.
Ce sont des réglages numériques de simulation, pas une calibration expérimentale des matériaux.
Voir la [documentation MuJoCo sur le glissement](https://mujoco.readthedocs.io/en/3.2.5/modeling.html).
Le pot conserve son articulation libre et sa masse de 0,603 kg ; aucune liaison rigide ne le fixe à la pince.


In [ ]:
%%writefile lift_metrics.py
"""Measure a real free-body lift independently of the gripper target."""
import numpy as np
import mujoco

class LiftMonitor:
    def __init__(self, model, data, object_name='mustard_bottle', table_name='table'):
        self.model, self.data = model, data
        self.body = model.body(object_name).id
        self.table = model.geom(table_name).id
        # Use the rendered bottle mesh, including the cap, for bottom clearance.
        self.visual = model.geom('mustard_bottle_visual_geom').id
        mesh = model.geom_dataid[self.visual]
        start, count = model.mesh_vertadr[mesh], model.mesh_vertnum[mesh]
        self.vertices = model.mesh_vert[start:start+count].copy()
        self.table_top = float(model.geom_pos[self.table,2] + model.geom_size[self.table,2])

    def sample(self, tcp_position, tcp_rotation):
        m,d = self.model,self.data
        rotation = d.geom_xmat[self.visual].reshape(3,3)
        bottom = float(np.min(self.vertices @ rotation[2,:]) + d.geom_xpos[self.visual,2])
        table_contact = False
        for c in d.contact:
            if self.table in (c.geom1,c.geom2):
                other = c.geom2 if c.geom1 == self.table else c.geom1
                if m.geom_bodyid[other] == self.body:
                    table_contact = True
        return dict(object_z_m=float(d.xpos[self.body,2]),
                    clearance_m=bottom-self.table_top,
                    object_in_tcp=(tcp_rotation.T @ (d.xpos[self.body]-tcp_position)).tolist(),
                    table_contact=table_contact)

def assess_lift(before, hold, final, *, required_rise=.10):
    """Require rise, table separation, persistent contact and a stable 3 s hold.

    hold contains timestamped samples, each with telemetry and contact forces.
    Heights are measured on the OBJECT, never inferred from a joint/TCP target.
    """
    if len(hold) < 2:
        raise ValueError('Need hold samples to evaluate lift stability.')
    offsets = np.array([row['object_in_tcp'] for row in hold])
    drift = float(np.max(np.linalg.norm(offsets-offsets[0],axis=1)))
    clearance = min(row['clearance_m'] for row in hold)
    rise = final['object_z_m']-before['object_z_m']
    separation = not any(row['table_contact'] for row in hold) and not final['table_contact']
    contact_fraction = float(np.mean([np.all(np.array(row['contact'])>.1) for row in hold]))
    duration = hold[-1]['time']-hold[0]['time']
    slip = float(np.linalg.norm(np.array(final['object_in_tcp'])-before['object_in_tcp']))
    success = (rise >= required_rise and clearance >= .08 and separation and
               contact_fraction >= .99 and drift <= .01 and duration >= 2.9)
    return dict(lift_success=bool(success), object_lift_m=float(rise),
                final_clearance_m=float(final['clearance_m']),
                min_hold_clearance_m=float(clearance), hold_relative_drift_m=drift,
                total_relative_slip_m=slip, hold_bilateral_contact_fraction=contact_fraction,
                hold_duration_measured_s=float(duration), table_separated=bool(separation))


In [ ]:
%%writefile panda_lab.py
"""Panda + French's mustard: pose IK and force-based finger impedance."""
from pathlib import Path
import copy
import json
import xml.etree.ElementTree as ET
import numpy as np
import mujoco
from scipy.spatial.transform import Rotation
from lift_metrics import LiftMonitor, assess_lift

HOME = np.array([0., -.4, 0., -2., 0., 1.6, .785398])
TABLE_Z = .4
OBJECT_XY = np.array([.48, 0.])
DOWN = np.array([[0., 1., 0.], [1., 0., 0.], [0., 0., -1.]])

def build_scene(root="assets", output="scene.xml"):
    root = Path(root).resolve()
    tree = ET.parse(root / "panda/panda.xml")
    xml = tree.getroot()
    xml.set("model", "Panda Frenchs laboratory")
    xml.find("compiler").attrib.pop("meshdir", None)
    for mesh in xml.findall("./asset/mesh"):
        mesh.set("file", (root / "panda/assets" / mesh.get("file")).as_posix())
    xml.find("option").attrib.update(timestep="0.002", gravity="0 0 -9.81", integrator="implicitfast",
                                     cone="elliptic", impratio="10", noslip_iterations="20")
    for tag in ["actuator", "tendon", "keyframe"]:
        node = xml.find(tag)
        if node is not None:
            xml.remove(node)
    actuator = ET.SubElement(xml, "actuator")
    for i, limit in enumerate([87, 87, 87, 87, 12, 12, 12], 1):
        ET.SubElement(actuator, "motor", name=f"arm_motor{i}", joint=f"joint{i}",
                      ctrlrange=f"{-limit} {limit}", gear="1")
    for i in [1, 2]:
        ET.SubElement(actuator, "motor", name=f"finger_motor{i}", joint=f"finger_joint{i}",
                      ctrlrange="-15 15", gear="1")
    world = xml.find("worldbody")
    world.find("body[@name='link0']").set("pos", f"0 0 {TABLE_Z}")
    hand = xml.find(".//body[@name='hand']")
    ET.SubElement(hand, "site", name="grasp_center", pos="0 0 0.1", size="0.004", rgba="0 1 1 1", group="4")
    # RGB axes are visual sites, so they do not change inertia or collisions.
    for axis, end, color in [("x", ".06 0 .1", "1 0 0 1"), ("y", "0 .06 .1", "0 1 0 1"),
                             ("z", "0 0 .16", "0 .3 1 1")]:
        ET.SubElement(hand, "site", name=f"tcp_{axis}", type="cylinder",
                      fromto=f"0 0 .1 {end}", size=".0015", rgba=color, group="4")
    ET.SubElement(world, "geom", name="floor", type="plane", size="2 2 .1", rgba=".18 .22 .28 1")
    ET.SubElement(world, "geom", name="table", type="box", pos=".3 0 .375", size=".7 .5 .025",
                  rgba=".62 .47 .32 1", friction="1 .005 .0001")
    for x in [-.3, .9]:
        for y in [-.4, .4]:
            ET.SubElement(world, "geom", type="box", pos=f"{x} {y} .175", size=".025 .025 .175",
                          rgba=".25 .27 .3 1")
    source = ET.parse(root / "mustard/source.xml").getroot()
    for asset in source.find("asset"):
        asset = copy.deepcopy(asset)
        if asset.get("file"):
            asset.set("file", (root / "mustard" / asset.get("file")).as_posix())
        xml.find("asset").append(asset)
    obj = copy.deepcopy(source.find("worldbody/body"))
    # Mesh is already upright (z is its long axis); keep real dimensions.
    vertices = np.array([list(map(float, line.split()[1:4])) for line in
                         (root / "mustard/mustard_bottle/textured.obj").read_text().splitlines()
                         if line.startswith("v ")])
    object_z = TABLE_Z - vertices[:, 2].min() + .001
    obj.set("pos", f"{OBJECT_XY[0]} {OBJECT_XY[1]} {object_z}")
    for geom in obj.findall("geom"):
        geom.set("friction", "1.2 .01 .001")
        geom.set("condim", "4")
    world.append(obj)
    ET.SubElement(world, "light", pos="1 -1 2.5", dir="-.3 .3 -1", diffuse=".7 .7 .7")
    visual = ET.SubElement(xml, "visual")
    ET.SubElement(visual, "global", offwidth="960", offheight="720")
    ET.indent(tree)
    tree.write(output, encoding="utf-8", xml_declaration=True)
    return Path(output)

class PandaLab:
    def __init__(self, scene="scene.xml"):
        self.model = m = mujoco.MjModel.from_xml_path(str(scene))
        self.data = d = mujoco.MjData(m)
        arm_j = np.array([m.joint(f"joint{i}").id for i in range(1, 8)])
        finger_j = np.array([m.joint(f"finger_joint{i}").id for i in [1, 2]])
        self.aq, self.av = m.jnt_qposadr[arm_j], m.jnt_dofadr[arm_j]
        self.fq, self.fv = m.jnt_qposadr[finger_j], m.jnt_dofadr[finger_j]
        self.au = np.array([m.actuator(f"arm_motor{i}").id for i in range(1, 8)])
        self.fu = np.array([m.actuator(f"finger_motor{i}").id for i in [1, 2]])
        self.bounds = m.jnt_range[arm_j].copy()
        self.site = m.site("grasp_center").id
        self.object_body = m.body("mustard_bottle").id
        self.lift_monitor = LiftMonitor(m, d)
        self.finger_bodies = [m.body(n).id for n in ["left_finger", "right_finger"]]
        self.kp_arm = np.array([400., 400., 350., 350., 180., 150., 100.])
        self.kd_arm = np.array([40., 40., 35., 35., 15., 12., 10.])
        self.k_finger, self.d_finger, self.force_limit = 400., 8., 12.
        self.log = []
        self.reset()

    def reset(self):
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[self.aq] = HOME
        self.data.qpos[self.fq] = .04
        mujoco.mj_forward(self.model, self.data)
        self.q_ref = HOME.copy()
        self.log = []

    def pose(self, data=None):
        d = self.data if data is None else data
        return d.site_xpos[self.site].copy(), d.site_xmat[self.site].reshape(3, 3).copy()

    def transform(self):
        p, r = self.pose()
        transform = np.eye(4)
        transform[:3, :3], transform[:3, 3] = r, p
        return transform

    def ik(self, position, rotation, seed=None, max_iterations=500):
        """6D damped least squares, world-frame Jacobian, bounded joints.

        Returns a configuration, never writes simulation qpos. Raises on failure.
        This is pose IK, not a collision-avoiding motion planner.
        """
        position, rotation = np.asarray(position, float), np.asarray(rotation, float)
        if position.shape != (3,) or rotation.shape != (3, 3) or not np.isfinite(position).all():
            raise ValueError("Expected position (3,) and rotation (3,3).")
        if not np.isfinite(rotation).all() or not np.allclose(rotation.T @ rotation, np.eye(3), atol=1e-6) or not np.isclose(np.linalg.det(rotation), 1):
            raise ValueError("rotation must be a proper rotation matrix")
        m = self.model
        work = mujoco.MjData(m)
        work.qpos[:] = self.data.qpos
        q = self.data.qpos[self.aq].copy() if seed is None else np.array(seed, float).copy()
        jp, jr = np.zeros((3, m.nv)), np.zeros((3, m.nv))
        for iteration in range(max_iterations):
            work.qpos[self.aq] = q
            mujoco.mj_forward(m, work)
            p, r = self.pose(work)
            ep = position - p
            er = Rotation.from_matrix(rotation @ r.T).as_rotvec()
            if np.linalg.norm(ep) < 2e-4 and np.linalg.norm(er) < 2e-3:
                return q
            mujoco.mj_jacSite(m, work, jp, jr, self.site)
            # 0.3 m/rad gives position and orientation comparable weights.
            jac = np.vstack([jp[:, self.av], .3 * jr[:, self.av]])
            err = np.r_[ep, .3 * er]
            delta = jac.T @ np.linalg.solve(jac @ jac.T + .02**2 * np.eye(6), err)
            delta *= min(1., .12 / max(np.linalg.norm(delta), 1e-12))
            q = np.clip(q + delta, self.bounds[:, 0] + .005, self.bounds[:, 1] - .005)
        raise RuntimeError(f"IK failed: position {np.linalg.norm(ep):.4f} m, rotation {np.linalg.norm(er):.4f} rad. Choose another pose.")

    def control(self, q_desired, finger_desired):
        """Arm torque PD + bias compensation; finger spring-damper in N."""
        m, d = self.model, self.data
        tau = self.kp_arm * (q_desired - d.qpos[self.aq]) - self.kd_arm * d.qvel[self.av] + d.qfrc_bias[self.av]
        d.ctrl[self.au] = np.clip(tau, m.actuator_ctrlrange[self.au, 0], m.actuator_ctrlrange[self.au, 1])
        # finger_desired is EACH finger displacement [m], not total opening.
        force = self.k_finger * (finger_desired - d.qpos[self.fq]) - self.d_finger * d.qvel[self.fv]
        d.ctrl[self.fu] = np.clip(force, -self.force_limit, self.force_limit)
        return d.ctrl[self.fu].copy()

    def contact_forces(self):
        """Sum normal contact forces [N] on each finger against the bottle."""
        result = np.zeros(2)
        wrench = np.zeros(6)
        for index in range(self.data.ncon):
            contact = self.data.contact[index]
            bodies = [self.model.geom_bodyid[contact.geom1], self.model.geom_bodyid[contact.geom2]]
            if self.object_body not in bodies:
                continue
            mujoco.mj_contactForce(self.model, self.data, index, wrench)
            for finger, body in enumerate(self.finger_bodies):
                if body in bodies:
                    result[finger] += max(wrench[0], 0.)
        return result

    def segment(self, q_target, duration, finger_start=.04, finger_end=.04, capture=None, phase="move"):
        q_start = self.q_ref.copy()
        steps = int(round(duration / self.model.opt.timestep))
        for step in range(steps):
            u = (step + 1) / steps
            blend = 10*u**3 - 15*u**4 + 6*u**5
            q_desired = q_start + blend * (q_target - q_start)
            f_desired = finger_start + blend * (finger_end - finger_start)
            force = self.control(q_desired, f_desired)
            mujoco.mj_step(self.model, self.data)
            if not np.isfinite(self.data.qpos).all():
                raise RuntimeError("Non-finite simulation state")
            if step % 20 == 0:
                mujoco.mj_forward(self.model, self.data)
                self.log.append(dict(time=float(self.data.time), phase=phase,
                    opening=float(self.data.qpos[self.fq].sum()), force=force.tolist(),
                    contact=self.contact_forces().tolist(), tcp=self.pose()[0].tolist(),
                    **self.lift_monitor.sample(*self.pose())))
            if capture is not None:
                capture(self)
        self.q_ref = np.array(q_target).copy()
        mujoco.mj_forward(self.model, self.data)

    def reach_pose(self, position, rotation, duration=3., capture=None):
        target = self.ik(position, rotation, seed=self.q_ref)
        self.segment(target, duration, capture=capture)
        self.segment(target, .5, capture=capture, phase="settle")
        p, r = self.pose()
        error = {"position_m": float(np.linalg.norm(np.array(position) - p)),
                 "orientation_rad": float(Rotation.from_matrix(np.array(rotation) @ r.T).magnitude())}
        if error["position_m"] > .01 or error["orientation_rad"] > .08:
            raise RuntimeError(f"Tracking failed (collision or torque saturation possible): {error}")
        return error

    def demo(self, capture=None):
        self.reset()
        self.segment(HOME, .5, capture=capture, phase="settle")
        grasp = np.r_[OBJECT_XY, TABLE_Z + .16]
        pregrasp = grasp + [0., 0., .20]
        self.reach_pose(pregrasp, DOWN, 3., capture)
        error = self.reach_pose(grasp, DOWN, 2., capture)
        self.segment(self.q_ref, 1.5, .04, 0., capture, "close")
        self.segment(self.q_ref, 1., 0., 0., capture, "hold")
        contacts = self.contact_forces()
        result = dict(pose_error=error, opening_m=float(self.data.qpos[self.fq].sum()),
                      finger_contact_N=contacts.tolist(), bilateral_contact=bool(np.all(contacts > .1)))
        if not result['bilateral_contact']:
            raise RuntimeError('No bilateral grasp: lift aborted.')
        before = self.lift_monitor.sample(*self.pose())
        # Keep the impedance closure active throughout lifting and holding.
        lift_target = grasp + [0., 0., .15]
        q_lift = self.ik(lift_target, DOWN, seed=self.q_ref)
        self.segment(q_lift, 4., 0., 0., capture, 'lift')
        self.segment(q_lift, 3., 0., 0., capture, 'hold_lift')
        final = self.lift_monitor.sample(*self.pose())
        result.update(assess_lift(before, [row for row in self.log if row['phase']=='hold_lift'], final))
        result['final_contact_N'] = self.contact_forces().tolist()
        result['final_tcp_error_m'] = float(np.linalg.norm(self.pose()[0]-lift_target))
        result["warnings"] = [int(w.number) for w in self.data.warning]
        return result

class Recorder:
    """Offscreen frames written directly to MP4 to avoid exhausting Colab RAM."""
    def __init__(self, model, path="panda_demo.mp4", fps=25):
        import imageio.v2 as imageio
        self.renderer = mujoco.Renderer(model, height=480, width=640)
        self.camera = mujoco.MjvCamera()
        self.camera.lookat[:] = [.35, 0., .65]
        self.camera.distance, self.camera.azimuth, self.camera.elevation = 1.7, 135, -25
        self.option = mujoco.MjvOption()
        self.option.geomgroup[3] = 0  # Hide collision proxies, keep textured visuals.
        self.option.sitegroup[4] = 1
        self.writer = imageio.get_writer(path, fps=fps, codec="libx264", quality=7)
        self.next_frame, self.fps = 0., fps

    def __call__(self, lab):
        if lab.data.time + 1e-9 >= self.next_frame:
            mujoco.mj_forward(lab.model, lab.data)
            self.renderer.update_scene(lab.data, camera=self.camera, scene_option=self.option)
            self.writer.append_data(self.renderer.render())
            self.next_frame += 1 / self.fps

    def close(self):
        self.writer.close()
        self.renderer.close()

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--video", action="store_true")
    args = parser.parse_args()
    build_scene()
    lab = PandaLab()
    recorder = Recorder(lab.model) if args.video else None
    try:
        result = lab.demo(recorder)
    finally:
        if recorder:
            recorder.close()
    Path("validation.json").write_text(json.dumps(result, indent=2))
    print(json.dumps(result, indent=2))


In [ ]:
from panda_lab import build_scene, PandaLab, Recorder, DOWN, TABLE_Z, OBJECT_XY
import numpy as np
import mujoco
print('MuJoCo', mujoco.__version__)
scene_path = build_scene(asset_root)
lab = PandaLab(scene_path)
print('Repère de pince : grasp_center, parent : hand, offset : [0, 0, 0.1] m')
print('T_world_tcp au repos :\n', lab.transform())
print('Articulations :', lab.model.nq, 'coordonnées ;', lab.model.nu, 'moteurs')

## 3. Démonstration : approche → fermeture → soulèvement → maintien

La cible est à 16 cm au-dessus du plateau pour éviter que la paume heurte le bouchon.
La préprise est 20 cm au-dessus. L’axe +z du TCP est orienté vers le bas, et son axe y
selon l’axe x du monde, le côté étroit du pot.

Une erreur de suivi trop grande interrompt la démonstration. Les résultats signalent
explicitement si les deux doigts sont en contact. Ensuite, une cible de montée de 15 cm est
suivie pendant 4 s, pince fermée, puis maintenue pendant 3 s. Le PD chargé peut laisser
quelques millimètres d’erreur : le déplacement du pot mesuré est d’environ 14 cm.
Relancer cette cellule réinitialise la scène.

`lift_success` exige une élévation réelle du pot d’au moins 10 cm, une garde à la table
d’au moins 8 cm durant le maintien, aucun contact avec la table, des contacts bilatéraux
sur au moins 99 % des échantillons, et moins de 1 cm de dérive relative pot/pince.


In [ ]:
from IPython.display import Video, display
import json
recorder = Recorder(lab.model, 'panda_demo.mp4')
try:
    results = lab.demo(capture=recorder)
finally:
    recorder.close()
print(json.dumps(results, indent=2))
display(Video('panda_demo.mp4', embed=True, width=720))
assert results['bilateral_contact'], 'Pas de contact bilatéral : inspecter la scène.'
assert results['lift_success'], 'Soulèvement ou maintien échoué : consulter les mesures.'
assert not any(results['warnings']), 'MuJoCo a signalé une instabilité.'

## 4. Observer l’impédance et les contacts

La commande moteur et la force normale de contact sont des grandeurs distinctes.
Une ouverture finale non nulle avec deux forces de contact positives montre que les doigts
rencontrent le pot. La quatrième courbe vérifie en plus que le bas du pot reste au-dessus
de la table. Le maintien est testé pendant 3 s, sans perturbation extérieure.


In [ ]:
import matplotlib.pyplot as plt
t = np.array([row['time'] for row in lab.log])
opening = np.array([row['opening'] for row in lab.log])
command = np.array([row['force'] for row in lab.log])
contact = np.array([row['contact'] for row in lab.log])
clearance = np.array([row['clearance_m'] for row in lab.log])
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
axes[0].plot(t, 1000 * opening)
axes[0].set_ylabel('Ouverture [mm]')
axes[1].plot(t, command[:, 0], label='Doigt 1')
axes[1].plot(t, command[:, 1], '--', label='Doigt 2')
axes[1].set_ylabel('Commande [N]')
axes[2].plot(t, contact[:, 0], label='Contact doigt 1')
axes[2].plot(t, contact[:, 1], '--', label='Contact doigt 2')
axes[2].set_ylabel('Force normale [N]')
axes[3].plot(t, 100 * clearance, label='Bas du pot au-dessus du plateau')
axes[3].axhline(0, color='black', linewidth=.8)
axes[3].set_ylabel('Garde [cm]'); axes[3].legend()
axes[3].set_xlabel('Temps [s]')
for ax in axes:
    ax.grid(alpha=.3)
axes[1].legend(); axes[2].legend()
plt.tight_layout(); plt.show()

## 5. Votre pose désirée

Exercice indépendant : réinitialiser puis atteindre une pose au-dessus de la table, pince ouverte.
Modifier `desired_position` et `desired_rotation`. L’orientation est une matrice 3 × 3
qui exprime les axes du TCP dans le monde. Une cible hors de portée déclenche une erreur.
Une pose atteignable ne garantit pas une trajectoire sans obstacle.


In [ ]:
from scipy.spatial.transform import Rotation
RUN_CUSTOM_POSE = False  # Passer à True pour lancer cet exercice.
desired_position = np.array([0.45, 0.10, 0.80])  # monde [m]
desired_rotation = DOWN.copy()
# Exemple de rotation supplémentaire autour de z MONDE :
# desired_rotation = Rotation.from_euler('z', 20, degrees=True).as_matrix() @ DOWN
if RUN_CUSTOM_POSE:
    lab.reset()
    recorder = Recorder(lab.model, 'custom_pose.mp4')
    try:
        error = lab.reach_pose(desired_position, desired_rotation, duration=3., capture=recorder)
        print('Erreur finale :', error)
    finally:
        recorder.close()
    display(Video('custom_pose.mp4', embed=True, width=720))

## 6. Paramètres de la fermeture et exercices

Modifier les gains puis relancer la démonstration (cellule 3) :
- `lab.k_finger = 200.` : comparer la force de contact et l’ouverture finale.
- `lab.d_finger = 12.` : observer l’amortissement.
- `lab.force_limit = 8.` : limiter la commande par doigt (rester entre 0 et 15 N, limite des moteurs XML).
- Comparer la pose de `hand` avec celle de `grasp_center` ; expliquer le décalage de 10 cm.
- Réduire la force de serrage et observer à partir de quand `lift_success` devient faux.
- Comparer la cible de montée (15 cm), l’élévation du pot et son glissement relatif.

**Lien futur avec Contact-GraspNet.** Une prédiction dans le repère caméra doit être
convertie : `T_world_tcp = T_world_camera @ T_camera_grasp @ T_grasp_tcp`.
Il faut identifier la convention exacte de la sortie du réseau et calibrer la dernière
transformation ; ne pas supposer que le repère prédit coïncide avec `hand` ou notre TCP.
Ce notebook ne charge ni poids du réseau ni nuage de points.

**Dépannage Colab.** Si OpenGL échoue après un import antérieur, redémarrer la session
et exécuter depuis le début. Si un téléchargement échoue, relancer sa cellule.
Les fichiers téléchargés sont mis en cache dans `assets/` pour la session.
La vidéo utilise un rendu logiciel ; sa génération peut prendre plus de temps que la simulation.

Sources : [API Python MuJoCo](https://mujoco.readthedocs.io/en/stable/python.html),
[Panda Menagerie](https://github.com/google-deepmind/mujoco_menagerie/tree/main/franka_emika_panda),
[YCB Dataset](https://github.com/elpis-lab/YCB_Dataset),
[Contact-GraspNet](https://github.com/NVlabs/contact_graspnet).
